In [2]:
import heapq
from collections import defaultdict

class UnionFind:
    def __init__(self, nodes):
        self.parent = {n: n for n in nodes}
        self.rank = {n: 0  for n in nodes}

    def find(self,x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra  == rb:
            return False
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        return True



def kruskal(nodes, edges):
    uf = UnionFind(nodes)
    mst_edges = []
    total_weights = 0

    for u, v, w in sorted(edges, key=lambda e: e[2]):
        if uf.union(u,v):
            mst_edges.append((u,v,w))
            total_weights += w
            if len(mst_edges) == len(nodes) -1:
                break
    return mst_edges, total_weights




def prim(nodes, edges, start=None):
    adj = defaultdict(list)
    for u,v, w in edges:
        adj[u].append((w,v))
        adj[v].append((w,u))

    start = start if start is not None else nodes[0]
    visited = {start}
    mst_edges = []
    total_weight = 0

    heap = [(w, start, v) for w, v in adj[start]]
    heapq.heapify(heap)

    while heap and len(visited) < len(nodes):
        w,u,v = heapq.heappop(heap)
        if v in visited:
            continue
        visited.add(v)
        mst_edges.append((u,v,w))
        total_weight += w
        for w2, nbr in adj[v]:
            if nbr not in visited:
                heapq.heappush(heap, (w2, v, nbr))

    return mst_edges, total_weight



if __name__ == "__main__":
    nodes = ["A","B","C","D","E","F"]
    edges = [
        ("A", "B", 7),
        ("A", "D", 5),
        ("B", "C", 8),
        ("B", "D", 9),
        ("B", "E", 7),
        ("C", "F", 5),
        ("D", "E", 15),
        ("E", "C", 8),
        ("E", "F", 9),
    ]


print("="*40)
print("Minimum Spanning Tree")
print("="*40)

mst_k, total_k = kruskal(nodes, edges)
print("\n Kruskal:")
for u,v,w in mst_k:
    print(f"{u} - {v} peso {w}")
print(f"pesos total {total_k}")


mst_p, total_p = prim(nodes, edges, start = "A")
print("\n prim inicio en A")
for u,v,w in mst_p:
    print(f"{u}-{v} peso {w}")
print(f"peso total {total_p}")

print("\n ambos producen el total" , total_k==total_p)

Minimum Spanning Tree

 Kruskal:
A - D peso 5
C - F peso 5
A - B peso 7
B - E peso 7
B - C peso 8
pesos total 32

 prim inicio en A
A-D peso 5
A-B peso 7
B-E peso 7
B-C peso 8
C-F peso 5
peso total 32

 ambos producen el total True


In [5]:
import heapq
from collections import Counter

class Node:
    def __init__(self, char, freq):
        self.char = char # Corrected typo: chair to char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq


def build_tree(text:str) -> Node:
    freq = Counter(text)
    heap = [Node(ch,f) for ch, f in freq.items()]
    heapq.heapify(heap)

    while len(heap) > 1: # Added missing colon
        left = heapq.heappop(heap)
        right = heapq.heappop(heap)
        merged = Node(None, left.freq + right.freq)
        merged.left = left
        merged.right = right
        heapq.heappush(heap, merged)

    return heap[0]


def build_codes(root: Node) -> dict[str, str]:
    codes = {}

    def traverse(node, prefix = ""):
        if node is None:
            return
        if node.char is not None:
            codes[node.char] = prefix or "0" # Assign '0' if prefix is empty (for single character texts)
        else:
            traverse(node.left, prefix + "0")
            traverse(node.right, prefix + "1")

    traverse(root)
    return codes





def encode(text: str) -> tuple[str, dict, Node]:
    root = build_tree(text)
    codes = build_codes(root)
    encoded = "".join(codes[ch] for ch in text) # Corrected typo: encodeed to encoded
    return encoded, codes, root


def decode(encoded: str, root: Node) -> str:
    result = []
    node = root
    for bit in encoded:
        node = node.left if bit == "0" else node.right
        if node.char is not None:
            result.append(node.char)
            node = root
    return "".join(result)




def compression_stats(text: str, encoded: str) -> None: # Corrected typo: textt to text
    original_bits = len(text) * 8
    compressed_bits = len(encoded)
    ratio = compressed_bits / original_bits * 100
    saving = 100 - ratio
    print(f" Bits originales: {original_bits}") # Changed 'originakes' to 'originales'
    print(f" Huffman bits: {compressed_bits}")
    print(f" Compression ratio: {ratio:.1f}%") # Changed label
    print(f" Savings: {saving:.1f}%") # Corrected typo: savings to saving and changed label



if __name__ == "__main__":
    text = "huffman encoding"

    print("="*52)
    print("Huffman encoding")
    print("="*52)
    print(f"\n Texto original: {text}") # Changed label

    encoded, codes, root = encode(text)

    print("\n Códigos Huffman (ordenados por frecuencia descendente):") # Changed label
    freq = Counter(text)
    # print(f"") # Removed empty print statement
    for ch, code in sorted(codes.items(), key=lambda x: -freq[x[0]]):
        label = repr(ch) if ch == " " else ch
        print(f"  '{label}': {code}") # Added print statement for codes


    preview = encoded[:60] + ("..." if len(encoded) > 60 else "") # Changed trailing character for clarity
    print(f"\n Bits codificados (preview): {preview}") # Changed label

    decoded = decode(encoded, root) # Corrected function call: decoded() to decode()
    print(f"\n Texto decodificado: {decoded}") # Changed label
    print(f" Coincide con original: {decoded == text}") # Corrected f-string syntax and changed label

    print("\n Estadísticas de compresión: ") # Changed label
    compression_stats(text, encoded)

Huffman encoding

 Texto original: huffman encoding

 Códigos Huffman (ordenados por frecuencia descendente):
  'n': 111
  'f': 001
  'e': 000
  'i': 0100
  'h': 0101
  'o': 0110
  'u': 0111
  '' '': 1000
  'g': 1001
  'c': 1010
  'm': 1011
  'd': 1100
  'a': 1101

 Bits codificados (preview): 0101011100100110111101111100000011110100110110001001111001

 Texto decodificado: huffman encoding
 Coincide con original: True

 Estadísticas de compresión: 
 Bits originales: 128
 Huffman bits: 58
 Compression ratio: 45.3%
 Savings: 54.7%


In [ ]:
from collections import defaultdict, deque

def topological_sort(nodes: list, edges: list[tuple] )  -> list:
    in_degree = {n: 0 for n in nodes}
    adj = defaultdict(list)

    for u, v, _ in edges:
        adj[u].append(v)
        in_degree[v] += 1

    queue = deque(n for n in nodes if in_degree[n] == 0)
    order = []

    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            in_degree[v] -= 1
            if in_degree[v] == 0:
                queue.append(v)

    if len(order) != len(nodes):
        raise ValueError("El grafo no es un DAG")

    return order



def dag_shortest_path(nodes: list, edges: list[tuple] , source) -> tuple[dict, dict]:

    adj = defaultdict(list)
    for u,v, w in edges:
        adj[u].append((v,w))

    INF = float("inf")
    dist = {n: INF for n in nodes}
    prev = {n: None for n in nodes}
    dist[source] = 0

    topo_order = topologicla_sort(nodes, edges)

    for u in topo_order:
        if dist[u] == INF:
            continue
        for v, w in adj[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u

    return dist, prev




def reconstruir_path(prev: dict, source, target) -> list | None:
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path_reverse()
    return path if path[0] == source else None



def dag_longest_path(nodes: list, edges: list[tuple], source) -> tuple[dict, dict]:

    neg_edges = [(u,v,-w) for u, v, w in edges]
    dist_neg, prev = dag_shortest_path(nodes, neg_edges, sources)
    dist = {n: -d for n, d in dist_neg.items()}
    return dist, prev

